# WESAD — 01 · Exploratory Data Analysis

**Project:** *When Wearable Stress Models Fail: Subject Generalization and Distribution Shift in Physiological Signals*

This notebook is the exploratory pass over the WESAD dataset. Its job is **not** to build a model —
it is to understand the data-generating process well enough to justify every modelling decision that follows.

The centrepiece is the **between-subject baseline analysis** (Section 5). If subjects sit at very different
physiological baselines, then:
- pooled random train/test splits leak subject identity → we need **leave-one-subject-out (LOSO)**,
- raw features confound *who the person is* with *what state they're in* → we need **subject normalization**,
- pooled regression coefficients can *reverse sign* vs. within-subject ones → **Simpson's paradox**,
- the correct hypothesis test is a **mixed-effects / repeated-measures** model, not pooled t-tests.

Everything downstream (the leakage audit, the confounding probe, the mixed-model statistics) is set up here.

> **Runs on Compute Canada.** Set `DATA_DIR` in the config cell to your WESAD root, then run top to bottom.
> Figures are saved to `outputs/figures/` and tables to `outputs/tables/`. Upload those back so the
> results can be wired into the final project package.

> **Naming caution:** "EDA" is overloaded here — it means *Exploratory Data Analysis* in this notebook's
> title, but *electrodermal activity* when it refers to a sensor channel. Code always uses the channel sense.

## 1 · Configuration

Only `DATA_DIR` should need changing. WESAD is expected as one folder per subject, each containing `SX.pkl`:

```
WESAD/
  S2/S2.pkl
  S3/S3.pkl
  ...
```

Subjects **S1** and **S12** were dropped by the dataset authors (sensor malfunction), leaving **15 subjects**.

In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)  # empty-slice means etc.

# >>> SET THIS to your WESAD root on Compute Canada <<<
DATA_DIR = Path("/home/USER/scratch/WESAD")   # each subject at DATA_DIR/SX/SX.pkl

# 15 valid subjects (S1, S12 discarded by dataset authors)
SUBJECTS = [f"S{i}" for i in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]]

# Sampling rates (Hz) per the WESAD readme
FS = {
    "label": 700,
    "chest": 700,          # ECG, EDA, EMG, Resp, Temp, ACC all 700 Hz
    "wrist_ACC": 32,
    "wrist_BVP": 64,
    "wrist_EDA": 4,
    "wrist_TEMP": 4,
}

# Protocol label IDs (readme II.3). 0 = transient, 5/6/7 = ignore.
LABEL_MAP = {0: "transient", 1: "baseline", 2: "stress", 3: "amusement",
             4: "meditation", 5: "ignore", 6: "ignore", 7: "ignore"}

# Conditions we actually analyse. We start with the 3 affect states; meditation kept for inventory.
CONDITIONS = {1: "baseline", 2: "stress", 3: "amusement", 4: "meditation"}
MODEL_STATES = {1: "baseline", 2: "stress", 3: "amusement"}  # what the classifier will see later

# Colours for consistent condition shading across plots
COND_COLOR = {"baseline": "#4C72B0", "stress": "#C44E52",
              "amusement": "#55A868", "meditation": "#8172B3"}

OUT = Path("outputs")
FIG = OUT / "figures"
TAB = OUT / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"{len(SUBJECTS)} subjects configured. Data root: {DATA_DIR}")

## 2 · Loading a subject

The `.pkl` files were pickled under Python 2, so Python 3 must load them with `encoding="latin1"` —
this is the single most common WESAD gotcha. Each file is a dict:

- `subject` — e.g. `"S2"`
- `signal["chest"]` — RespiBAN: `ACC (n,3)`, `ECG`, `EDA`, `EMG`, `Resp`, `Temp` (all `n,1` @ 700 Hz)
- `signal["wrist"]` — Empatica E4: `ACC (n,3)` @32, `BVP` @64, `EDA` @4, `TEMP` @4
- `label` — protocol condition per sample @700 Hz (aligned to the chest/RespiBAN clock)

Because labels live on the 700 Hz clock but wrist signals are slower, we map labels to any signal's
own timebase by time, rather than by a fixed stride (a stride only works cleanly for the 4 Hz channels).

In [ ]:
def load_subject(subj, data_dir=DATA_DIR):
    """Load one WESAD subject dict (Py2 pickle -> latin1)."""
    path = Path(data_dir) / subj / f"{subj}.pkl"
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")


def labels_for(data, fs_signal, n):
    """Return the 700 Hz protocol label resampled onto a signal of length n sampled at fs_signal.
    Maps each signal sample to the nearest label sample by time (works for any rate)."""
    lab = np.asarray(data["label"]).ravel()
    t = np.arange(n) / fs_signal                       # seconds for each signal sample
    idx = np.round(t * FS["label"]).astype(int)
    idx = np.clip(idx, 0, len(lab) - 1)
    return lab[idx]


def get_channel(data, location, channel):
    """Return a 1-D signal (ACC collapsed to magnitude) and its sampling rate."""
    sig = np.asarray(data["signal"][location][channel], dtype=float)
    if channel == "ACC":
        sig = np.sqrt((sig ** 2).sum(axis=1))          # 3-axis -> magnitude
        fs = FS[f"{location}_ACC"] if location == "wrist" else FS["chest"]
    else:
        sig = sig.ravel()
        fs = FS[f"{location}_{channel}"] if f"{location}_{channel}" in FS else FS["chest"]
    return sig, fs

### 2.1 · Structural inventory of one subject
Confirm the dict layout, channel shapes, and label IDs match the readme before trusting anything else.

In [ ]:
d0 = load_subject(SUBJECTS[0])

print("top-level keys :", list(d0.keys()))
print("subject id     :", d0["subject"])
print("label length   :", np.asarray(d0["label"]).shape,
      "| unique ids:", sorted(np.unique(d0["label"]).tolist()))
print()
for loc in ["chest", "wrist"]:
    print(f"[{loc}]")
    for ch, arr in d0["signal"][loc].items():
        arr = np.asarray(arr)
        secs = arr.shape[0] / (FS.get(f"{loc}_{ch}", FS["chest"]))
        print(f"   {ch:5s} shape={str(arr.shape):12s} ~{secs/60:5.1f} min")

## 3 · Integrity sweep across all subjects

Load every subject once, verify the expected channels exist, and record signal durations. Any subject that
fails to load or is missing channels is flagged rather than silently dropped — data quality is part of the story
(the readme notes per-subject quality caveats).

In [ ]:
expected = {
    "chest": ["ACC", "ECG", "EDA", "EMG", "Resp", "Temp"],
    "wrist": ["ACC", "BVP", "EDA", "TEMP"],
}

rows, problems = [], []
for s in SUBJECTS:
    try:
        d = load_subject(s)
    except Exception as e:
        problems.append((s, f"load failed: {e}"))
        continue
    for loc, chans in expected.items():
        for ch in chans:
            if ch not in d["signal"][loc]:
                problems.append((s, f"missing {loc}/{ch}"))
    lab = np.asarray(d["label"])
    dur_min = len(lab) / FS["label"] / 60
    eda = np.asarray(d["signal"]["wrist"]["EDA"]).ravel()
    rows.append(dict(subject=s, protocol_min=dur_min,
                     wrist_EDA_min=len(eda) / FS["wrist_EDA"] / 60,
                     label_ids=",".join(map(str, sorted(np.unique(lab).tolist())))))

inv = pd.DataFrame(rows).set_index("subject")
print("Problems:", problems if problems else "none")
inv

## 4 · Protocol structure & class balance

WESAD is a **designed within-subject experiment**: every subject passes through the same conditions, but the
*durations differ* (baseline and meditation are long; amusement is short). That imbalance matters for both
modelling (class weights, metric choice) and for the statistics (unequal within-condition sample sizes).

We measure seconds spent in each condition per subject.

In [ ]:
dur_rows = []
for s in SUBJECTS:
    d = load_subject(s)
    lab = np.asarray(d["label"]).ravel()
    for cid, cname in CONDITIONS.items():
        secs = int((lab == cid).sum()) / FS["label"]
        dur_rows.append(dict(subject=s, condition=cname, seconds=secs))

dur = pd.DataFrame(dur_rows)
dur_wide = dur.pivot(index="subject", columns="condition", values="seconds").reindex(SUBJECTS)
dur_wide = dur_wide[[c for c in ["baseline", "stress", "amusement", "meditation"] if c in dur_wide]]
dur_wide.to_csv(TAB / "condition_durations_seconds.csv")

print("Median seconds per condition:")
print(dur_wide.median().round(1))

ax = dur_wide.plot(kind="bar", stacked=True, figsize=(11, 4),
                   color=[COND_COLOR[c] for c in dur_wide.columns])
ax.set_ylabel("seconds"); ax.set_title("Time per condition, per subject (design imbalance)")
ax.legend(title="condition", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.savefig(FIG / "condition_durations.png", dpi=140); plt.show()

## 5 · Raw wrist signals over the protocol (one subject)

A sanity plot: the three wrist channels we'll build features from — **EDA** (electrodermal, tonic arousal),
**TEMP** (skin temperature, peripheral vasoconstriction), and **ACC magnitude** (motion) — across the whole
session, shaded by condition. Watch the ACC track: if motion differs systematically by condition, any
"stress" model risks becoming a **motion detector** — the confound we test explicitly in a later notebook.

In [ ]:
def plot_subject_timeline(subj, save=True):
    d = load_subject(subj)
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
    for ax, ch in zip(axes, ["EDA", "TEMP", "ACC"]):
        sig, fs = get_channel(d, "wrist", ch)
        lab = labels_for(d, fs, len(sig))
        t = np.arange(len(sig)) / fs / 60  # minutes
        ax.plot(t, sig, lw=0.6, color="0.2")
        # shade condition spans
        for cid, cname in CONDITIONS.items():
            mask = lab == cid
            if mask.any():
                ax.fill_between(t, sig.min(), sig.max(), where=mask,
                                color=COND_COLOR[cname], alpha=0.18, step="mid")
        ax.set_ylabel(f"wrist {ch}")
    axes[-1].set_xlabel("time (min)")
    axes[0].set_title(f"{subj}: wrist signals over protocol "
                      f"(shaded = " + ", ".join(f"{c}" for c in COND_COLOR) + ")")
    plt.tight_layout()
    if save:
        plt.savefig(FIG / f"timeline_{subj}.png", dpi=140)
    plt.show()

plot_subject_timeline(SUBJECTS[0])

## 6 · Between-subject baseline structure — the core motivating result

**This is the section that justifies the whole project design.**

If we look only at the **baseline** condition (subjects at rest, same instruction), do they sit at the same
physiological level? If not — if subject A's resting EDA is systematically higher than subject B's — then a
model trained on pooled data can classify state partly by *recognising the person*. Random splits hide this;
LOSO exposes it.

We quantify it with an **intraclass-correlation-style ratio**:

$$\text{ICC-like} = \frac{\text{between-subject variance of baseline means}}{\text{between-subject variance} + \text{mean within-subject variance}}$$

Values near 1 → baseline is dominated by *who the subject is*; near 0 → subjects are interchangeable.
High values are the empirical licence for subject-normalization and LOSO.

In [ ]:
def baseline_samples(subj, location, channel):
    d = load_subject(subj)
    sig, fs = get_channel(d, location, channel)
    lab = labels_for(d, fs, len(sig))
    return sig[lab == 1]  # condition 1 = baseline


def icc_like(per_subject_arrays):
    """between-subject variance of means / (that + mean within-subject variance)."""
    means = np.array([a.mean() for a in per_subject_arrays if len(a)])
    within = np.array([a.var(ddof=1) for a in per_subject_arrays if len(a) > 1])
    between_var = means.var(ddof=1)
    within_var = within.mean()
    return between_var / (between_var + within_var), between_var, within_var


channels = [("wrist", "EDA"), ("wrist", "TEMP"), ("wrist", "ACC")]

fig, axes = plt.subplots(1, len(channels), figsize=(14, 4.5))
summary = []
for ax, (loc, ch) in zip(axes, channels):
    arrs = [baseline_samples(s, loc, ch) for s in SUBJECTS]
    # downsample points for a readable boxplot but keep full stats
    ax.boxplot([a for a in arrs], labels=[s.replace("S", "") for s in SUBJECTS],
               showfliers=False)
    ax.set_title(f"{loc} {ch} @ baseline")
    ax.set_xlabel("subject"); ax.tick_params(axis="x", labelsize=7)
    icc, bv, wv = icc_like(arrs)
    summary.append(dict(location=loc, channel=ch, icc_like=icc,
                        between_var=bv, within_var=wv))
axes[0].set_ylabel("signal value")
plt.suptitle("Baseline distribution per subject — spread of the boxes = between-subject structure")
plt.tight_layout(); plt.savefig(FIG / "baseline_between_subject.png", dpi=140); plt.show()

icc_df = pd.DataFrame(summary)
icc_df.to_csv(TAB / "baseline_icc_like.csv", index=False)
print(icc_df)
print("\nReading: ICC-like near 1 => baseline is dominated by subject identity.")
print("=> justifies LOSO, subject-normalization, and mixed models over pooled tests.")

## 7 · Do condition effects look the same across subjects?

Now the within-subject picture: for each subject, the mean of each channel per condition. Two things to look for:

1. **Consistent direction** — does stress raise EDA for (almost) everyone? Consistency across subjects is what
   a mixed model with a random intercept per subject is built to capture.
2. **Crossing lines / different offsets** — subjects starting at different baselines, with slopes that vary,
   is precisely the geometry that produces **Simpson's paradox** when data is pooled. This plot is the visual
   pre-registration of that experiment.

We save the per-subject × per-condition means to CSV — this table is the direct input to the mixed-effects
ANOVA and the Simpson's-paradox analysis in the statistics notebook.

In [ ]:
rows = []
for s in SUBJECTS:
    d = load_subject(s)
    for loc, ch in channels:
        sig, fs = get_channel(d, loc, ch)
        lab = labels_for(d, fs, len(sig))
        for cid, cname in MODEL_STATES.items():
            v = sig[lab == cid]
            if len(v):
                rows.append(dict(subject=s, location=loc, channel=ch,
                                 condition=cname, mean=v.mean(),
                                 std=v.std(ddof=1) if len(v) > 1 else np.nan,
                                 n=len(v)))

feat = pd.DataFrame(rows)
feat.to_csv(TAB / "subject_condition_means.csv", index=False)

order = ["baseline", "stress", "amusement"]
fig, axes = plt.subplots(1, len(channels), figsize=(14, 4.5))
for ax, (loc, ch) in zip(axes, channels):
    sub = feat[(feat.location == loc) & (feat.channel == ch)]
    for s in SUBJECTS:
        line = sub[sub.subject == s].set_index("condition")["mean"].reindex(order)
        ax.plot(order, line.values, marker="o", lw=1, alpha=0.6)
    ax.set_title(f"{loc} {ch}: per-subject condition means")
    ax.set_xlabel("condition")
axes[0].set_ylabel("mean signal")
plt.suptitle("Each line is one subject — crossing / offset lines foreshadow Simpson's paradox")
plt.tight_layout(); plt.savefig(FIG / "per_subject_condition_means.png", dpi=140); plt.show()

feat.head(9)

## 8 · Takeaways → decisions carried into the next notebooks

Fill the bracketed numbers in from your run — they become the evidence you cite in the write-up and CV.

- **Between-subject baseline structure is [strong / weak]** (ICC-like ≈ `[__]` for wrist EDA).
  → Validation must be **LOSO**, not random split; features should be **subject-normalized**; hypothesis
  tests must use **subject as a random effect**.
- **Condition classes are imbalanced** (amusement ≪ baseline).
  → Report balanced metrics (macro-F1 / balanced accuracy), not raw accuracy.
- **Motion (ACC) [does / does not] track condition.**
  → Schedule the **confounding probe** (ACC ablation) — is the model a stress detector or a motion detector?
- **Per-subject condition lines [cross / are parallel].**
  → Sets up the **Simpson's-paradox** experiment (pooled vs. within-subject coefficient signs).

**Artifacts written for downstream use / upload:**
- `outputs/tables/subject_condition_means.csv` — input to the mixed-model ANOVA & Simpson's analysis
- `outputs/tables/baseline_icc_like.csv` — the between-subject quantification
- `outputs/tables/condition_durations_seconds.csv` — class-balance record
- `outputs/figures/*.png` — the figures above

**Next notebook (`02_...`)** implements the *five resume-value pillars*: the leakage audit, the confounding
probe, LOSO with confidence intervals (the SLURM sweep), uncertainty/calibration, and the explicit
"what to do next" tied to Eli's longitudinal problem. **Then** the statistics notebook (`03_...`) implements
the mixed-effects ANOVA + FDR + Simpson's-paradox design against `subject_condition_means.csv`.